# Notebook 1: Dataset Preparation

This notebook scans the CASIA v1.0 and v2.0 image datasets, extracts labels from filenames,
validates image integrity, and creates the 70/15/15 train/validation/test split CSV that
all subsequent notebooks depend on.

**Prerequisites**:
- CASIA v1.0 dataset uploaded to Colab (see Cell 4 for upload instructions)
- CASIA v2.0 dataset uploaded to Colab (see Cell 4 for upload instructions)

**Outputs**:
- `data/metadata/dataset_split.csv` — the authoritative train/val/test split
- `data/metadata/split_info.json` — summary statistics for the split

## Step 1: Clone the Repository and Install Dependencies

Replace the placeholder URL with your actual GitHub repository URL before running.

In [1]:
# Set your GitHub repository URL
GITHUB_REPO_URL = "https://github.com/Salman-ahmed-2/image-manipulation-detection.git"
REPO_NAME = "image-manipulation-detection"

import os
from pathlib import Path

# Check if we are already in the repo or need to clone (Colab environment)
if Path("config.py").exists() and Path("data").exists():
    print("Already in the repository root:", os.getcwd())
elif Path(f"/content/{REPO_NAME}").exists():
    os.chdir(f"/content/{REPO_NAME}")
    print("Changed directory to repo root:", os.getcwd())
elif os.path.exists("/content"):
    # Standard Google Colab environment
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        print(f"Cloning {GITHUB_REPO_URL} into /content/{REPO_NAME} ...")
        os.system(f"git clone {GITHUB_REPO_URL} {REPO_NAME}")
    os.chdir(REPO_NAME)
    print("Current working directory:", os.getcwd())
else:
    print("Local environment detected:", os.getcwd())


Local environment detected: /media/salman/p5/5th sem/ai/Project_ai/Project_semester/image-manipulation-detection/image-manipulation-detection/notebooks/colab


## Step 2: GPU / Environment Check

Dataset preparation is CPU-bound, so GPU is not required here, but we confirm the
environment is set up correctly.

In [2]:
import platform
import sys

print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    gpus = tf.config.list_physical_devices("GPU")
    print("Available GPUs:", gpus if gpus else "None (CPU only — OK for dataset prep)")
except ImportError:
    print("TensorFlow not yet installed — will be available after requirements install.")

try:
    import PIL
    print("Pillow version:", PIL.__version__)
except ImportError:
    print("Pillow not yet installed.")

print("\nEnvironment check complete.")

Python version: 3.14.4 (main, Aug 20 2026, 10:41:58) [GCC 15.2.0]
Platform: Linux-7.0.0-44-generic-x86_64-with-glibc2.43


I0000 00:00:1795762336.950752  122504 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795762337.782282  122504 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795762349.082359  122504 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.22.0-rc0
Available GPUs: None (CPU only — OK for dataset prep)
Pillow version: 13.0.0

Environment check complete.


E0000 00:00:1795762366.571556  122504 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



## Step 3: Dataset Path Configuration

**Dataset folder layout on your project:**

Your repository expects image files placed inside `data/raw/`:
```text
data/
└── raw/
    ├── casia_v1/             <-- Put CASIA v1.0 images here (Au_*.jpg, Tp_*.jpg)
    └── casia_v2/             <-- Put CASIA v2.0 images here (Au_*.jpg, Tp_*.tif/png)
```

**How to supply your dataset images to Colab:**

**Option A — Upload a zip archive and unzip directly into data/raw/:**
```bash
# If you uploaded casia_v2.zip or casia_v1.zip to Colab's file panel:
!unzip -q /content/casia_v2.zip -d data/raw/casia_v2/
!unzip -q /content/casia_v1.zip -d data/raw/casia_v1/
```

**Option B — Mount Google Drive (if your images are stored on Google Drive):**
```python
from google.colab import drive
drive.mount('/content/drive')
# Copy to project data/raw/ or update CASIA_V2_PATH in the cell below to point to Drive
# !cp -r "/content/drive/MyDrive/casia_v2/" data/raw/
```

**Option C — Custom Path Override:**
If your dataset is already extracted in a custom folder (e.g. `/content/casia2/`), simply update `CASIA_V1_PATH` or `CASIA_V2_PATH` in the code cell below.


In [ ]:
import sys
import os
import zipfile
from pathlib import Path

# Handle deleted CWD in Jupyter kernel safely
try:
    current_dir = Path.cwd().resolve()
except (FileNotFoundError, OSError):
    try:
        current_dir = Path(__file__).resolve().parent
    except NameError:
        current_dir = Path.cwd().resolve()

# Walk up to find the root folder containing config.py and data/
repo_root = current_dir
while repo_root != repo_root.parent:
    if (repo_root / "config.py").exists() and (repo_root / "data").exists():
        break
    repo_root = repo_root.parent

os.chdir(str(repo_root))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import Config

CASIA_V1_PATH = Config.CASIA_V1_DIR
CASIA_V2_PATH = Config.CASIA_V2_DIR
METADATA_DIR = Config.METADATA_DIR
SPLIT_CSV_PATH = Config.DATASET_SPLIT_PATH

METADATA_DIR.mkdir(parents=True, exist_ok=True)
CASIA_V1_PATH.mkdir(parents=True, exist_ok=True)
CASIA_V2_PATH.mkdir(parents=True, exist_ok=True)

v1_count = len([p for p in CASIA_V1_PATH.rglob("*") if p.is_file() and not p.name.startswith(".")]) if CASIA_V1_PATH.is_dir() else 0
v2_count = len([p for p in CASIA_V2_PATH.rglob("*") if p.is_file() and not p.name.startswith(".")]) if CASIA_V2_PATH.is_dir() else 0

# Auto-unzip helper: automatically detect and extract dataset ZIP if folders are empty
if v1_count == 0 and v2_count == 0:
    search_dirs = [Config.RAW_DATA_DIR, Config.DATA_DIR, repo_root]
    zip_files = []
    seen = set()
    for d in search_dirs:
        if d.is_dir():
            for z in sorted(d.glob("*.zip")):
                if z.resolve() not in seen:
                    seen.add(z.resolve())
                    zip_files.append(z)
    
    if zip_files:
        print(f"Found {len(zip_files)} dataset archive(s) to auto-extract:")
        for zpath in zip_files:
            print(f"  -> Extracting {zpath.name} ({zpath.stat().st_size / (1024*1024):.1f} MB)...", flush=True)
            with zipfile.ZipFile(zpath, "r") as archive:
                names = archive.namelist()
                roots = {n.split("/")[0] for n in names if "/" in n}
                if "casia_v1" in roots or "casia_v2" in roots:
                    archive.extractall(Config.RAW_DATA_DIR)
                elif "v1" in zpath.name.lower() or "casia1" in zpath.name.lower():
                    archive.extractall(CASIA_V1_PATH)
                else:
                    target = CASIA_V2_PATH if ("v2" in zpath.name.lower() or "casia2" in zpath.name.lower()) else Config.RAW_DATA_DIR
                    archive.extractall(target)
            print(f"     Finished extracting {zpath.name}.")
        
        # Recount after unzipping
        v1_count = len([p for p in CASIA_V1_PATH.rglob("*") if p.is_file() and not p.name.startswith(".")]) if CASIA_V1_PATH.is_dir() else 0
        v2_count = len([p for p in CASIA_V2_PATH.rglob("*") if p.is_file() and not p.name.startswith(".")]) if CASIA_V2_PATH.is_dir() else 0

print("Project Root  :", repo_root)
print(f"CASIA v1 path : {CASIA_V1_PATH} (exists: {CASIA_V1_PATH.is_dir()}, image files: {v1_count})")
print(f"CASIA v2 path : {CASIA_V2_PATH} (exists: {CASIA_V2_PATH.is_dir()}, image files: {v2_count})")
print("Metadata dir  :", METADATA_DIR)
print("Split CSV path:", SPLIT_CSV_PATH)

if v1_count == 0 and v2_count == 0:
    print("\n" + "="*60)
    print("NOTICE: No image files found in data/raw/casia_v1 or data/raw/casia_v2.")
    print("Place your dataset ZIP into the project root, data/, or data/raw/ and re-run this cell.")
    print("="*60)
else:
    print(f"\nTotal image files ready to process: {v1_count + v2_count}")


## Step 4: Import Project Modules

We import the project's own dataset utilities rather than re-implementing them here.
This ensures the notebook stays in sync with the production codebase.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# Project modules
from config import Config
from src.data.dataset_loader import extract_label_from_filename
from src.data.dataset_validator import scan_dataset_directory
from src.data.dataset_splitter import DatasetSplitter, verify_no_leakage

print("All imports successful.")
print(f"Config.INPUT_SHAPE         = {Config.INPUT_SHAPE}")
print(f"Config.AUTHENTIC_PREFIX    = {Config.AUTHENTIC_PREFIX!r}")
print(f"Config.TAMPERED_PREFIX     = {Config.TAMPERED_PREFIX!r}")
print(f"Config.DATASET_TRAIN_RATIO = {Config.DATASET_TRAIN_RATIO}")
print(f"Config.DATASET_VAL_RATIO   = {Config.DATASET_VALIDATION_RATIO}")
print(f"Config.DATASET_TEST_RATIO  = {Config.DATASET_TEST_RATIO}")

## Step 5: Scan Dataset Directories

`scan_dataset_directory` recursively walks each CASIA directory, opens every image to
verify it can be decoded, and returns a list of metadata dictionaries. Corrupt or
unreadable images are flagged with `is_valid=False` rather than raising an exception.

In [ ]:
print("Scanning CASIA v1 ...", flush=True)
v1_records = scan_dataset_directory(CASIA_V1_PATH)
print(f"  Found {len(v1_records)} image(s) in CASIA v1")

print("Scanning CASIA v2 ...", flush=True)
v2_records = scan_dataset_directory(CASIA_V2_PATH)
print(f"  Found {len(v2_records)} image(s) in CASIA v2")

if not v1_records and not v2_records:
    print("\nNo images found in either dataset directory.")
    print("Please upload the CASIA dataset images following the instructions in Step 3.")
else:
    total = len(v1_records) + len(v2_records)
    print(f"\nTotal images discovered: {total}")

# Tag each record with its dataset version
for rec in v1_records:
    rec["dataset_version"] = Config.CASIA_V1_VERSION
for rec in v2_records:
    rec["dataset_version"] = Config.CASIA_V2_VERSION

all_records = v1_records + v2_records
inventory_df = pd.DataFrame(all_records)
print(f"\nInventory DataFrame shape: {inventory_df.shape}")
if not inventory_df.empty:
    display(inventory_df.head(5))

## Step 6: Extract Labels from Filenames

Per the CASIA labelling convention, the filename prefix determines the class:
- `Au_` prefix → label 0 (Authentic)
- `Tp_` prefix → label 1 (Manipulated)

Files that match neither prefix are logged as warnings and skipped.

In [ ]:
if inventory_df.empty:
    print("No images to label — skipping. Upload CASIA images first.")
    labeled_df = pd.DataFrame()
else:
    records = []
    unlabeled_count = 0
    for _, row in inventory_df.iterrows():
        try:
            info = extract_label_from_filename(row["filename"])
            rec = dict(row)
            rec["label"] = info["label"]
            rec["label_name"] = info["label_name"]
            rec["manipulation_type"] = info["manipulation_type"]
            rec["group_id"] = info["group_id"]
            records.append(rec)
        except ValueError:
            unlabeled_count += 1

    labeled_df = pd.DataFrame(records)
    if not labeled_df.empty:
        labeled_df["label"] = labeled_df["label"].astype(int)

    print(f"Total images scanned : {len(inventory_df)}")
    print(f"Successfully labelled: {len(labeled_df)}")
    print(f"Unlabelled / skipped : {unlabeled_count}")
    if not labeled_df.empty:
        display(labeled_df[["filename", "label", "label_name", "manipulation_type", "group_id", "dataset_version"]].head(10))


## Step 7: Validation Summary — Class Distribution

We check for class balance and flag any potential issues before splitting.

In [ ]:
import matplotlib.pyplot as plt

if labeled_df.empty:
    print("No labelled data available. Upload CASIA images first.")
else:
    class_counts = labeled_df["label_name"].value_counts()
    print("=" * 40)
    print("CLASS DISTRIBUTION SUMMARY")
    print("=" * 40)
    for label_name, count in class_counts.items():
        pct = 100 * count / len(labeled_df)
        print(f"  {label_name:<25} {count:>6} ({pct:.1f}%)")
    print(f"  {'TOTAL':<25} {len(labeled_df):>6}")

    # Per-dataset breakdown
    print("\nPer-dataset breakdown:")
    print(labeled_df.groupby(["dataset_version", "label_name"]).size().to_string())

    # Quick bar chart
    fig, ax = plt.subplots(figsize=(6, 4))
    class_counts.plot(kind="bar", ax=ax, color=["steelblue", "tomato"], edgecolor="black")
    ax.set_title("Class Distribution")
    ax.set_xlabel("Class")
    ax.set_ylabel("Image Count")
    ax.tick_params(axis="x", rotation=0)
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 10,
            str(int(bar.get_height())),
            ha="center", va="bottom", fontsize=11
        )
    plt.tight_layout()
    plt.show()

## Step 8: Invalid Image Report

Any images that could not be opened or verified are shown here.
These are excluded from training automatically.

In [ ]:
if inventory_df.empty:
    print("No inventory data available.")
else:
    invalid_df = inventory_df[inventory_df["is_valid"] == False]
    if invalid_df.empty:
        print("All images passed integrity check. No corrupt files found.")
    else:
        print(f"Found {len(invalid_df)} corrupt or unreadable image(s):")
        display(invalid_df[["filename", "filepath", "dataset_version"]])
        print("\nThese files will be excluded from the dataset split.")

    # Keep only valid images for splitting
    if "is_valid" in labeled_df.columns:
        before = len(labeled_df)
        labeled_df = labeled_df[labeled_df["is_valid"] == True].copy()
        after = len(labeled_df)
        print(f"\nValid images for splitting: {after} (removed {before - after} invalid)")

## Step 9: Create the Dataset Split (70 / 15 / 15)

We use `DatasetSplitter` which performs group-aware splitting via `GroupShuffleSplit`.
The `group_id` column ensures that images from the same donor image never appear in
both training and evaluation partitions (preventing data leakage).

If `group_id` is not in the inventory (CASIA v1 may not have it), we derive one
from the filename prefix to simulate group identity.

In [ ]:
if labeled_df.empty:
    print("No data to split. Upload CASIA images and re-run from Step 5.")
    split_df = pd.DataFrame()
else:
    # Derive group_id if not present: use filename stem minus the numeric suffix
    # e.g. 'Au_ani10' -> 'Au_ani', 'Tp_D_NRN_S_B_ani10054_ani10075_10054' -> prefix
    if "group_id" not in labeled_df.columns:
        def _derive_group(filename: str) -> str:
            """Derive a coarse group id from a CASIA filename."""
            stem = Path(filename).stem
            # Strip trailing digits to group images by scene/content
            stripped = stem.rstrip("0123456789")
            return stripped if stripped else stem

        labeled_df = labeled_df.copy()
        labeled_df["group_id"] = labeled_df["filename"].apply(_derive_group)
        print(f"Derived {labeled_df['group_id'].nunique()} groups from filenames.")

    splitter = DatasetSplitter(
        train_ratio=Config.DATASET_TRAIN_RATIO,
        val_ratio=Config.DATASET_VALIDATION_RATIO,
        test_ratio=Config.DATASET_TEST_RATIO,
        random_state=Config.RANDOM_SEED,
    )

    try:
        split_df = splitter.split_dataframe(labeled_df, group_col="group_id")
        print("Dataset split created successfully.")
        print("\nSplit distribution:")
        print(split_df["split"].value_counts().to_string())
        print("\nSplit + Class breakdown:")
        print(split_df.groupby(["split", "label_name"]).size().to_string())
    except ValueError as exc:
        print(f"Split failed: {exc}")
        print("This usually means too few unique groups. Check your dataset.")
        split_df = pd.DataFrame()

## Step 10: Verify No Data Leakage

This assertion guarantees that no donor group appears in more than one split partition.
If this check fails, the split must not be used for training.

In [ ]:
if split_df.empty:
    print("No split to verify — skipping leakage check.")
else:
    try:
        no_leakage = verify_no_leakage(split_df, group_col="group_id")
        print("✓ Leakage verification PASSED: no group spans multiple split partitions.")
    except AssertionError as exc:
        print(f"✗ LEAKAGE DETECTED: {exc}")
        print("Do NOT use this split for training. Investigate the group_id assignment.")
        raise

## Step 11: Save Metadata Outputs

We save:
1. `dataset_split.csv` — the full split DataFrame (read by Notebooks 2 and 3)
2. `split_info.json` — a human-readable summary for quick reference

In [ ]:
import json

if split_df.empty:
    print("No split data to save. Run Steps 5-10 with actual CASIA images first.")
else:
    # Save the split CSV
    SPLIT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    split_df.to_csv(SPLIT_CSV_PATH, index=False)
    print(f"Saved dataset split CSV to: {SPLIT_CSV_PATH}")
    print(f"  Rows: {len(split_df)}")
    print(f"  Columns: {list(split_df.columns)}")

    # Build split info summary
    split_counts = split_df["split"].value_counts().to_dict()
    class_counts_per_split = (
        split_df.groupby(["split", "label_name"])
        .size()
        .reset_index(name="count")
        .to_dict(orient="records")
    )
    split_info = {
        "total_images": len(split_df),
        "split_counts": split_counts,
        "class_distribution_per_split": class_counts_per_split,
        "train_ratio": Config.DATASET_TRAIN_RATIO,
        "validation_ratio": Config.DATASET_VALIDATION_RATIO,
        "test_ratio": Config.DATASET_TEST_RATIO,
        "random_seed": Config.RANDOM_SEED,
        "leakage_verified": True,
    }

    split_info_path = METADATA_DIR / "split_info.json"
    with split_info_path.open("w", encoding="utf-8") as fh:
        json.dump(split_info, fh, indent=2)
    print(f"\nSaved split info JSON to: {split_info_path}")
    print(json.dumps(split_info, indent=2))

## Summary

This notebook has:
1. Scanned CASIA v1 and v2 raw directories using `scan_dataset_directory()`
2. Extracted binary labels (0 = Authentic, 1 = Manipulated) from filenames
3. Flagged and excluded corrupt/unreadable images
4. Created a 70/15/15 group-aware train/validation/test split
5. Verified no donor group spans multiple partitions (no data leakage)
6. Saved `data/metadata/dataset_split.csv` and `split_info.json`

**Next step**: Run `02_mobilenetv2_training.ipynb` to train the model on this split.